# Day 1 | Lab 1.1: Hello-LLM Client (Multi-Provider API Patterns)

**Duration:** ~1.5 hours

**Scenario.** Domain-mixed (insurance claims, hospital discharge notes, e-commerce, banking FAQs, fraud-alert messaging) — preserved from the source notebook so trainees see how the *same* API patterns work across domains. The training goal is multi-provider fluency, not domain immersion.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Set up and call **five** LLM providers programmatically — OpenAI, Anthropic, Google Gemini, Groq Cloud, and **AWS Bedrock** — using their official Python SDKs.
2. Distinguish reasoning models (`gpt-5-mini`) from non-reasoning models (`gpt-4.1-mini`) and pick the right parameter (`reasoning_effort` vs `temperature`).
3. Control generation parameters across providers — `temperature`, `max_tokens`, `reasoning_effort`.
4. Build a **provider-agnostic wrapper** so application code can swap models without rewrites.
5. Make a Bedrock call two ways — native `boto3.converse` (production AWS pattern) and `ChatBedrock` (LangChain abstraction we adopt in Module 3).

**Models in scope**

| Provider | Model | Type | API Style |
|---|---|---|---|
| OpenAI | `gpt-5-mini` | Proprietary (Reasoning) | Responses API |
| OpenAI | `gpt-4.1-mini` | Proprietary (Non-reasoning) | Responses API |
| Anthropic | `claude-haiku-4-5` | Proprietary | Messages API |
| Google | `gemini-2.5-flash` | Proprietary (Thinking) | Google GenAI SDK |
| Groq Cloud | `llama-3.3-70b-versatile` | Open-source (Llama 3.3) | OpenAI-compatible SDK |
| **AWS Bedrock** | `anthropic.claude-haiku-4-5-*` | Proprietary (managed) | `boto3` Converse API + `ChatBedrock` |

---


In [1]:
import sys
print(sys.executable)

c:\venvs\eclerx-genai\Scripts\python.exe


## 1. Install Dependencies

In [3]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
!pip install openai anthropic google-genai groq boto3 
# 'langchain>=1.0' 'langchain-core>=1.0' 'langchain-aws>=0.2'


  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached websockets-16.0-cp312-cp312-win_amd64.whl.metadata (7.0 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached certifi-2026.4.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Usin


[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
%%capture
pip install langchain>=1.0 langchain-core>=1.0 langchain-aws>=0.2

## 2. API Key Configuration

We load API keys from **Google Colab's Secrets Manager**.

> **Setup:** Click the 🔑 icon in the left sidebar → Add each key → Toggle "Notebook access" ON.

In [15]:
!pip install python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv(r"C:\Users\prash\OneDrive\2026\Training\eClerx-GenAI-Training\.env")
except ImportError:
    pass

# Verify keys are loaded (prints status, never prints actual values)
for key in ['OPENAI_API_KEY', 'GROQ_API_KEY', 'GEMINI_API_KEY', 'ANTHROPIC_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


OPENAI_API_KEY: ✅ Loaded
GROQ_API_KEY: ✅ Loaded
GEMINI_API_KEY: ✅ Loaded
ANTHROPIC_API_KEY: ✅ Loaded


## 3. Initialize API Clients

In [19]:
from openai import OpenAI
from anthropic import Anthropic
from google import genai
from groq import Groq

# Each client picks up its API key from the corresponding environment variable
openai_client = OpenAI()                # reads OPENAI_API_KEY
anthropic_client = Anthropic()          # reads ANTHROPIC_API_KEY
gemini_client = genai.Client()          # reads GEMINI_API_KEY
groq_client = Groq()                    # reads GROQ_API_KEY

print("All 4 API clients initialized ✅")

All 4 API clients initialized ✅


---
## 4. Task 1 — OpenAI: `gpt-5-mini` via the Responses API

### Business Scenario: Insurance Claim Summary
An insurance company wants to auto-generate concise claim summaries from adjuster notes. We use OpenAI's latest **Responses API** — the current standard that replaces the older Chat Completions API.

**Key points about `gpt-5-mini`:**
- It is a **reasoning model** — it thinks step-by-step internally before answering
- It does **NOT** support the `temperature` parameter (only `reasoning_effort`)
- Uses `instructions` for system-level guidance and `input` for user messages
- Output accessed via `response.output_text`

In [20]:
import time

# --- OpenAI gpt-5-mini via Responses API ---
instructions = "You are a senior insurance claims analyst. Summarize claim details concisely."

user_input = """Claim #INS-2025-78432: Policyholder Ramesh Gupta reported water damage to his
ground-floor retail shop on Jan 15, 2025 due to a burst municipal pipeline. Estimated damage:
₹8.5 lakhs covering inventory loss (₹5L), structural repairs (₹2.5L), and business interruption
for 12 days (₹1L). Adjuster visited on Jan 18. Photos and municipal complaint letter attached.
Policy covers flood/water damage up to ₹15L with ₹50K deductible."""

start = time.time()
response = openai_client.responses.create(
    model="gpt-5-mini",
    instructions=instructions,          # system-level guidance
    input=user_input,                    # user message (string or list of messages)
    reasoning={"effort": "low"},         # reasoning depth: low / medium / high
    max_output_tokens=300,
)
latency = round(time.time() - start, 2)

print(f"Provider : OpenAI (gpt-5-mini)")
print(f"Latency  : {latency}s")
print(f"Tokens   : {response.usage.input_tokens} in / {response.usage.output_tokens} out")
print(f"\n--- Response ---\n{response.output_text}")

Provider : OpenAI (gpt-5-mini)
Latency  : 8.17s
Tokens   : 139 in / 264 out

--- Response ---
Claim summary — INS-2025-78432

- Policyholder: Ramesh Gupta
- Loss date/report: 15 Jan 2025
- Location: Ground-floor retail shop
- Cause: Burst municipal pipeline (municipal complaint letter attached)
- Adjuster inspection: 18 Jan 2025
- Documentation: Photos and municipal complaint letter attached

Estimated loss (total ₹8.5 lakh):
- Inventory loss: ₹5,00,000
- Structural repairs: ₹2,50,000
- Business interruption (12 days): ₹1,00,000

Policy limits/terms:
- Coverage: Flood/water damage up to ₹15,00,000
- Deductible: ₹50,000

Net payable (pre-validation): Estimated gross ₹8,50,000 − ₹50,000 deductible = ₹8,00,000 (subject to validation of losses, policy terms, and any sublimits/exclusions).


---
## 5. Task 2 — Anthropic: `claude-haiku-4-5` via Messages API

### Business Scenario: Patient Discharge Instructions
A hospital system wants to auto-generate easy-to-understand discharge notes from clinical summaries. Anthropic's API has a distinct pattern — the `system` prompt is a **separate parameter**, not inside the messages array.

In [21]:
# --- Anthropic claude-haiku-4-5 via Messages API ---
system_prompt = "You are a patient-friendly medical communicator. Explain in simple, reassuring language."

user_message = """Patient: Mrs. Sunita Devi, 58F. Admitted for elective laparoscopic cholecystectomy.
Procedure uneventful. Post-op vitals stable. Tolerating oral diet. Discharged on Day 2.
Rx: Tab Paracetamol 650mg TDS x 5 days, Tab Pantoprazole 40mg OD x 7 days.
Follow-up with surgeon in 10 days. Watch for fever >101°F, wound redness, or persistent vomiting."""

start = time.time()
response = anthropic_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=400,
    system=system_prompt,                # Anthropic: system is a separate parameter
    messages=[{"role": "user", "content": user_message}],
    temperature=0.3,                     # Claude supports temperature 0-1
)
latency = round(time.time() - start, 2)

print(f"Provider : Anthropic (claude-haiku-4-5)")
print(f"Latency  : {latency}s")
print(f"Tokens   : {response.usage.input_tokens} in / {response.usage.output_tokens} out")
print(f"\n--- Response ---\n{response.content[0].text}")

Provider : Anthropic (claude-haiku-4-5)
Latency  : 4.38s
Tokens   : 154 in / 376 out

--- Response ---
# Your Discharge Summary - Simple Explanation

Dear Mrs. Devi,

Congratulations! Your gallbladder surgery went very well. Here's what you need to know:

## ✅ What Happened
- Your gallbladder was successfully removed using a minimally invasive (keyhole) method
- Your body is recovering well
- You're eating normally, which is a great sign

## 💊 Your Medicines (Take at Home)

**Paracetamol 650mg**
- For pain relief
- Take **3 times daily** for **5 days**
- Take with food if your stomach feels upset

**Pantoprazole 40mg**
- Protects your stomach
- Take **once daily** for **7 days**
- Take in the morning, preferably before breakfast

## 🏥 Follow-up Care
- **See your surgeon in 10 days** for a check-up
- This is important—don't miss this appointment

## ⚠️ When to Seek Immediate Help
Call your doctor or go to the hospital if you notice:
- **Fever above 101°F** (high temperature)
- **Redness

---
## 6. Task 3 — Google Gemini: `gemini-2.5-flash` via GenAI SDK

### Business Scenario: E-Commerce Product Description
An online retailer wants to auto-generate compelling product descriptions from raw product specs. Gemini uses the `google-genai` SDK with `system_instruction` inside the `config` dict.

In [22]:
# --- Google Gemini via google-genai SDK ---
system_instruction = "You are a creative e-commerce copywriter. Write engaging product descriptions."

user_message = """Product: Borosil 500ml Vacuum Insulated Flask
Material: 304 stainless steel, BPA-free lid
Keeps hot: 12 hours | Keeps cold: 24 hours
Colors: Midnight Black, Ocean Blue, Rose Gold
Weight: 320g | Price: ₹1,299
Target: Urban professionals, gym-goers, college students"""

start = time.time()
response = gemini_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=user_message,
    config={
        "system_instruction": system_instruction,
        "temperature": 0.7,
        "max_output_tokens": 400,
    }
)
latency = round(time.time() - start, 2)

print(f"Provider : Google Gemini (gemini-2.5-flash)")
print(f"Latency  : {latency}s")
print(f"Tokens   : {response.usage_metadata.prompt_token_count} in / {response.usage_metadata.candidates_token_count} out")
print(f"\n--- Response ---\n{response.text}")

Provider : Google Gemini (gemini-2.5-flash)
Latency  : 3.78s
Tokens   : 101 in / 16 out

--- Response ---
Here are a few options, playing with slightly different angles for an engaging description:


---
## 7. Task 4 — Groq Cloud: `llama-3.3-70b-versatile` (Open-Source LLM)

### Business Scenario: Banking FAQ Bot
A retail bank needs a quick FAQ responder for common customer queries. Groq Cloud runs open-source models (Llama 3.3) on their custom **LPU hardware**, delivering extremely low latency.

Groq uses an **OpenAI-compatible SDK** — same `chat.completions.create` interface, just a different client.

In [23]:
# --- Groq Cloud (Llama 3.3 70B) via OpenAI-compatible SDK ---
system_prompt = "You are a helpful retail banking assistant. Be concise and accurate."

user_message = """What is the difference between a savings account and a fixed deposit?
Which one should a salaried employee with ₹2 lakhs surplus choose for 1 year?"""

start = time.time()
response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ],
    temperature=0.5,
    max_tokens=400,
)
latency = round(time.time() - start, 2)

print(f"Provider : Groq Cloud (llama-3.3-70b-versatile)")
print(f"Latency  : {latency}s")
print(f"Tokens   : {response.usage.prompt_tokens} in / {response.usage.completion_tokens} out")
print(f"\n--- Response ---\n{response.choices[0].message.content}")

Provider : Groq Cloud (llama-3.3-70b-versatile)
Latency  : 1.07s
Tokens   : 81 in / 158 out

--- Response ---
**Key differences:**

1. **Liquidity**: Savings account - easily accessible, Fixed Deposit (FD) - locked for a fixed term.
2. **Interest rate**: FD typically offers higher interest rates than savings accounts.
3. **Flexibility**: Savings account allows withdrawals and deposits, FD has penalties for early withdrawals.

**Recommendation for a salaried employee with ₹2 lakhs surplus for 1 year:**
Consider a Fixed Deposit (FD) for a 1-year term, as it offers a higher interest rate (around 5-6% pa) and is a low-risk investment. This will help you earn a higher return on your surplus funds. However, if you need easy access to your money, a savings account might be a better option.


---
## 8. Task 5 — Unified Wrapper Functions (Provider-Agnostic Pattern)

In production, you want to abstract away provider-specific APIs behind a **common interface**. This lets you swap providers without rewriting application code — a critical pattern for cost optimization and reliability.

Below, we wrap each provider into a function with the same signature: `(system_prompt, user_prompt, **kwargs) → dict`

In [24]:
def call_openai(system_prompt, user_prompt, model="gpt-5-mini", max_tokens=500, **kwargs):
    """
    Call OpenAI Responses API.
    For reasoning models (gpt-5-*): uses reasoning_effort instead of temperature.
    For non-reasoning models (gpt-4.1-*): uses temperature.
    """
    start = time.time()
    api_kwargs = {
        "model": model,
        "instructions": system_prompt,
        "input": user_prompt,
        "max_output_tokens": max_tokens,
    }
    # Reasoning models use reasoning_effort; non-reasoning use temperature
    if model.startswith("gpt-5"):
        api_kwargs["reasoning"] = {"effort": kwargs.get("reasoning_effort", "low")}
    else:
        api_kwargs["temperature"] = kwargs.get("temperature", 0.7)

    response = openai_client.responses.create(**api_kwargs)
    latency = round(time.time() - start, 2)

    return {
        "provider": "OpenAI",
        "model": model,
        "response": response.output_text,
        "latency_sec": latency,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }


def call_anthropic(system_prompt, user_prompt, model="claude-haiku-4-5",
                    max_tokens=500, **kwargs):
    """Call Anthropic Messages API. System prompt is a separate parameter."""
    start = time.time()
    response = anthropic_client.messages.create(
        model=model,
        max_tokens=max_tokens,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}],
        temperature=kwargs.get("temperature", 0.7),
    )
    latency = round(time.time() - start, 2)

    return {
        "provider": "Anthropic",
        "model": model,
        "response": response.content[0].text,
        "latency_sec": latency,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }


def call_gemini(system_prompt, user_prompt, model="gemini-2.5-flash",
                max_tokens=500, **kwargs):
    """Call Google Gemini via google-genai SDK."""
    start = time.time()
    response = gemini_client.models.generate_content(
        model=model,
        contents=user_prompt,
        config={
            "system_instruction": system_prompt,
            "temperature": kwargs.get("temperature", 0.7),
            "max_output_tokens": max_tokens,
        }
    )
    latency = round(time.time() - start, 2)

    return {
        "provider": "Google Gemini",
        "model": model,
        "response": response.text,
        "latency_sec": latency,
        "input_tokens": response.usage_metadata.prompt_token_count,
        "output_tokens": response.usage_metadata.candidates_token_count,
    }


def call_groq(system_prompt, user_prompt, model="llama-3.3-70b-versatile",
              max_tokens=500, **kwargs):
    """Call Groq Cloud — OpenAI-compatible SDK running Llama on LPU hardware."""
    start = time.time()
    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=kwargs.get("temperature", 0.7),
        max_tokens=max_tokens,
    )
    latency = round(time.time() - start, 2)

    return {
        "provider": "Groq (Llama 3.3)",
        "model": model,
        "response": response.choices[0].message.content,
        "latency_sec": latency,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
    }


print("Unified wrapper functions defined ✅")

Unified wrapper functions defined ✅


### 8a. Side-by-Side Comparison — Same Prompt, All 4 Providers

**Business Scenario: Credit Card Fraud Alert Explanation**

A bank wants to send customers a clear explanation when a transaction is flagged. Let's see how each model handles the same prompt.

In [25]:
system = "You are a bank's fraud detection assistant. Explain clearly and calmly to the customer."

prompt = """A transaction of ₹47,500 at 'ELECMART ONLINE' was flagged as suspicious on your
HDFC Visa card ending 4821. The flag was triggered because: (1) it's 3x your average online
spend, (2) the merchant is new to your profile, (3) the transaction happened at 2:47 AM IST.
Explain this to the customer and tell them what to do next. Keep it under 100 words."""

# Run all 4 providers
providers = [
    ("OpenAI (gpt-5-mini)",  call_openai),
    ("Anthropic (Claude)",    call_anthropic),
    ("Google Gemini",         call_gemini),
    ("Groq (Llama 3.3)",     call_groq),
]

results = []
for name, fn in providers:
    print(f"⏳ Calling {name}...")
    try:
        result = fn(system, prompt)
        results.append(result)
        print(f"   ✅ {result['latency_sec']}s | {result['output_tokens']} tokens")
    except Exception as e:
        print(f"   ❌ Failed: {e}")

⏳ Calling OpenAI (gpt-5-mini)...
   ✅ 4.95s | 250 tokens
⏳ Calling Anthropic (Claude)...
   ✅ 2.34s | 158 tokens
⏳ Calling Google Gemini...
   ✅ 4.23s | 19 tokens
⏳ Calling Groq (Llama 3.3)...
   ✅ 0.6s | 66 tokens


In [26]:
# Display all responses side-by-side
for r in results:
    print(f"\n{'━' * 70}")
    print(f"🤖 {r['provider']} ({r['model']}) — {r['latency_sec']}s | {r['output_tokens']} tokens")
    print(f"{'━' * 70}")
    print(r["response"])


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖 OpenAI (gpt-5-mini) — 4.95s | 250 tokens
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
We flagged a ₹47,500 charge at "ELECMART ONLINE" on your HDFC Visa ending 4821 because (1) it’s ~3x your usual online spend, (2) this merchant is new on your account, and (3) it occurred at 2:47 AM IST — all typical fraud signals.

Please review: if you recognize the purchase, reply here or confirm via the number on your card. If unauthorized, immediately block the card via phone/app and report the transaction so we can raise a dispute and issue a replacement.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖 Anthropic (claude-haiku-4-5) — 2.34s | 158 tokens
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Transaction Alert - HDFC Visa 4821

We've flagged a ₹47,500 charge at ELECMART ONLINE as potentially suspicious due to three factors:

1. **High amount*

### 📝 Discussion Points
- Which model respects the **100-word limit** best?
- Which tone feels most appropriate for a real bank notification?
- Notice the **latency differences** — Groq (LPU hardware) is typically the fastest
- gpt-5-mini may take slightly longer due to internal reasoning steps

---
## 8b. Adding AWS Bedrock as the 5th Provider

### Why a 5th provider — and why Bedrock?
Banks, insurers, and healthcare clients increasingly require LLMs to be served from inside their own AWS account for compliance:
- **VPC-private networking** — request payloads never leave the AWS network.
- **IAM-based access control** — no per-engineer API keys; permissions ride on the EC2 / Lambda role.
- **Region-pinned endpoints** — data-residency for regulated workloads.
- **Single AWS bill** across Anthropic, Meta, Mistral, Cohere, and Amazon Titan models.

We'll call **Claude Haiku 4.5 on Bedrock** two ways:
1. **Native `boto3.client('bedrock-runtime').converse(...)`** — the lowest-level AWS-native call you'd put in a Lambda or EC2 service.
2. **`ChatBedrock` LangChain wrapper** — a 5-line preview of the LangChain abstraction we'll adopt in Module 3.

> **Model-ID note.** AWS rotates Bedrock model versions. The IDs below were current at build time — verify against [AWS Bedrock — Supported foundation models](https://docs.aws.amazon.com/bedrock/latest/userguide/models-supported.html) before running. Some regions also require the **cross-region inference profile** form (e.g., `us.anthropic.claude-haiku-4-5-...`) — see the AWS console for what's enabled in your account.


In [27]:
# --- AWS credentials (env vars or IAM role; no Colab) ---
import os

aws_region = os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')
print(f'AWS region              : {aws_region}')
print(f"AWS_ACCESS_KEY_ID set   : {'✅' if os.environ.get('AWS_ACCESS_KEY_ID') else '❌ (relying on IAM role / shared config)'}")


AWS region              : us-east-1
AWS_ACCESS_KEY_ID set   : ✅


### 8b.1. Native `boto3` Call — the AWS Production Pattern

The `bedrock-runtime` client exposes the **Converse API** — a unified message-based interface across every model on Bedrock (Anthropic, Meta, Mistral, Cohere, Titan). The same code works whether `modelId` points to Claude, Llama, or Titan.


In [30]:
# --- Bedrock via native boto3 (Claude Haiku 4.5) ---
import boto3
import time

bedrock = boto3.client('bedrock', region_name=aws_region)
bedrock_runtime = boto3.client('bedrock-runtime', region_name=aws_region)

print("✅ AWS Bedrock clients initialized")

# Test Bedrock access by listing available models
try:
    response = bedrock.list_foundation_models()
    available_models = [m['modelId'] for m in response['modelSummaries']]

    # Filter for models we'll use
    titan_models = [m for m in available_models if 'titan-embed' in m]
    nova_models = [m for m in available_models if 'nova' in m]

    print(f"\n✅ Access to {len(available_models)} Bedrock models")
    print(f"   Titan Embedding models: {len(titan_models)}")
    print(f"   Nova models: {len(nova_models)}")

    if titan_models:
        print(f"\n   Using Titan: {titan_models[0]}")
    if nova_models:
        print(f"   Using Nova: {[m for m in nova_models if 'nova-pro' in m][0] if any('nova' in m for m in nova_models) else nova_models[0]}")

except Exception as e:
    print(f"⚠️  Could not list models: {str(e)}")
    print("   This is OK - we'll proceed with known model IDs")

✅ AWS Bedrock clients initialized

✅ Access to 125 Bedrock models
   Titan Embedding models: 7
   Nova models: 22

   Using Titan: amazon.titan-embed-g1-text-02
   Using Nova: amazon.nova-pro-v1:0


In [31]:
bedrock.list_foundation_models()

{'ResponseMetadata': {'RequestId': 'e0610de7-27c9-4436-a943-ab6de70cea4b',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 29 Apr 2026 01:32:32 GMT',
   'content-type': 'application/json',
   'content-length': '161992',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'e0610de7-27c9-4436-a943-ab6de70cea4b'},
  'RetryAttempts': 0},
 'modelSummaries': [{'modelArn': 'arn:aws:bedrock:us-east-1::foundation-model/nvidia.nemotron-nano-12b-v2',
   'modelId': 'nvidia.nemotron-nano-12b-v2',
   'modelName': 'NVIDIA Nemotron Nano 12B v2 VL BF16',
   'providerName': 'NVIDIA',
   'inputModalities': ['TEXT', 'IMAGE'],
   'outputModalities': ['TEXT'],
   'responseStreamingSupported': True,
   'customizationsSupported': [],
   'inferenceTypesSupported': ['ON_DEMAND'],
   'modelLifecycle': {'status': 'ACTIVE',
    'startOfLifeTime': datetime.datetime(2025, 12, 2, 0, 0, tzinfo=tzutc())}},
  {'modelArn': 'arn:aws:bedrock:us-east-1::foundation-model/qwen.qwen3-coder-next',
   'modelId': 'qwen

# `bedrock` vs `bedrock-runtime` in boto3

These are **two separate AWS services** with distinct purposes:

## `bedrock` — Control Plane

Use for **management and configuration** operations (infrequent, admin-style calls):

- `list_foundation_models()` — discover available models
- `get_foundation_model()` — model metadata
- Manage **custom models** (fine-tuning jobs, model customization)
- Manage **provisioned throughput**
- Manage **guardrails**, **model evaluation jobs**, **knowledge bases** (Agents for Bedrock)
- IAM/access configuration, model invocation logging

```python
bedrock = boto3.client("bedrock", region_name="us-east-1")
models = bedrock.list_foundation_models()
```

## `bedrock-runtime` — Data Plane

Use for **actually invoking models** (the hot path in your app):

- `invoke_model()` — single request/response
- `invoke_model_with_response_stream()` — streaming tokens
- `converse()` — unified conversational API across model families (recommended)
- `converse_stream()` — streaming variant of converse

```python
runtime = boto3.client("bedrock-runtime", region_name="us-east-1")
resp = runtime.converse(
    modelId="anthropic.claude-sonnet-4-5-20250929-v1:0",
    messages=[{"role": "user", "content": [{"text": "Hello"}]}],
)
```

## Quick rule of thumb

| You want to...                                  | Client             |
| ----------------------------------------------- | ------------------ |
| Send a prompt / get a completion                | `bedrock-runtime`  |
| Stream tokens                                   | `bedrock-runtime`  |
| List or describe models                         | `bedrock`          |
| Create/manage fine-tuned models, guardrails, KBs | `bedrock`         |

There are also two more siblings worth knowing:

- `bedrock-agent` — control plane for Agents/Knowledge Bases (create, update)
- `bedrock-agent-runtime` — invoke agents and retrieve from knowledge bases

Same pattern: plain name = manage it, `-runtime` = call it.

In [ ]:
# Initialize Amazon Titan Text G1 Express (instead of Claude)
# Available Titan Text models:
#   - amazon.titan-text-express-v1 (faster, cheaper)
#   - amazon.titan-text-lite-v1 (fastest, cheapest)
#   - amazon.titan-text-premier-v1:0 (most capable)

In [ ]:

# Verify the latest dated suffix in your AWS account before running.
# BEDROCK_MODEL_ID = 'anthropic.claude-haiku-4-5-20251001-v1:0'
BEDROCK_MODEL_ID = "amazon.nova-pro-v1:0"

system_prompt = 'You are a helpful retail banking assistant. Be concise and accurate.'
user_message = (
    'In 2 sentences, explain why a regulated bank would route Claude requests '
    'through Amazon Bedrock instead of calling the Anthropic API directly.'
)

start = time.time()
response = bedrock_runtime.converse(
    modelId=BEDROCK_MODEL_ID,
    system=[{'text': system_prompt}],
    messages=[{'role': 'user', 'content': [{'text': user_message}]}],
    inferenceConfig={'maxTokens': 400, 'temperature': 0.5},
)
latency = round(time.time() - start, 2)

output_text = response['output']['message']['content'][0]['text']
usage = response['usage']

print(f'Provider : AWS Bedrock ({BEDROCK_MODEL_ID})')
print(f'Latency  : {latency}s')
print(f"Tokens   : {usage['inputTokens']} in / {usage['outputTokens']} out")
print(f'Response :\n{output_text}')


Provider : AWS Bedrock (amazon.nova-pro-v1:0)
Latency  : 2.75s
Tokens   : 41 in / 34 out
Response :
For compliance and security reasons, a regulated bank would prefer using Amazon Bedrock, which offers governed access to AI services, over direct API calls to Anthropic.


### 8b.2. `ChatBedrock` — A Module 3 Preview

Native `boto3.converse(...)` gives you full control but uses an AWS-specific request shape. **`ChatBedrock`** wraps that same call in LangChain's universal `.invoke([SystemMessage, HumanMessage])` interface — identical to `ChatAnthropic`, `ChatOpenAI`, etc. Once your application code calls `.invoke(...)`, switching providers becomes a one-line change.

We'll build whole chains on top of `ChatBedrock` in Module 3. Treat the cell below as the **5-line preview**.


In [35]:
# --- Bedrock via ChatBedrock (LangChain wrapper) ---
from langchain_aws import ChatBedrock
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatBedrock(
    model_id=BEDROCK_MODEL_ID,
    region_name=aws_region,
    model_kwargs={'max_tokens': 400, 'temperature': 0.5},
)

lc_response = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content=user_message),
])

print('LangChain ChatBedrock response:')
print(lc_response.content)
print(f'\nUsage metadata: {lc_response.usage_metadata}')


LangChain ChatBedrock response:
To ensure compliance with data privacy regulations and to leverage Amazon's robust security infrastructure.

Usage metadata: {'input_tokens': 41, 'output_tokens': 18, 'total_tokens': 59, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}}


### 📝 Why we showed both
- The **`boto3` native call** is what you'll write in production Lambdas and EC2 services — closest to the metal, no Python framework overhead, AWS SDK is already in every Lambda runtime.
- The **`ChatBedrock` wrapper** uses LangChain's universal chat-model interface — every downstream chain (RAG, agents, multi-step orchestration) takes a `BaseChatModel` and doesn't care which provider is plugged in. You'll see this pattern repeatedly from Module 3 onward.
- **Decision rule:** wrap with LangChain when the call sits inside a chain or agent; use raw `boto3` when the call is the *only* LLM step in a tightly-scoped Lambda.


---
## 9. Task 6 — Temperature Control: Creativity vs. Determinism

### Why This Matters
Temperature is the primary knob for controlling **randomness** in LLM outputs:
- **Low temperature (0.1–0.3):** Deterministic, focused — good for factual tasks, data extraction, structured outputs
- **High temperature (0.8–1.0):** Creative, diverse — good for brainstorming, marketing copy, creative writing

**Important:** Reasoning models like `gpt-5-mini` do **NOT** support temperature. For temperature demos, we use the non-reasoning models:

| Provider | Model for Temperature Demo |
|----------|---------------------------|
| OpenAI | `gpt-4.1-mini` (non-reasoning, supports temperature) |
| Anthropic | `claude-haiku-4-5` (supports temperature 0–1) |
| Google | `gemini-2.5-flash` (supports temperature) |
| Groq | `llama-3.3-70b-versatile` (supports temperature 0–2) |

### Business Scenario: Marketing Tagline Generation
A fintech startup needs taglines for their new UPI payments app called **"PayZap"**. The marketing team wants both safe/predictable options and bold/creative options.

In [36]:
# --- Temperature comparison: Low (safe) vs High (creative) ---
creative_system = "You are a creative copywriter for a fintech startup."
creative_prompt = "Write 3 catchy taglines (max 10 words each) for a UPI payments app called 'PayZap'."

# Define providers with models that support temperature
temp_providers = [
    ("OpenAI (gpt-4.1-mini)",     call_openai,     {"model": "gpt-4.1-mini"}),
    ("Anthropic (Claude Haiku)",   call_anthropic,  {}),
    ("Google Gemini",              call_gemini,     {}),
    ("Groq (Llama 3.3)",          call_groq,       {}),
]

print("🧊 LOW TEMPERATURE (0.2) — Safe / Predictable")
print("=" * 60)
for name, fn, extra_kwargs in temp_providers:
    try:
        r = fn(creative_system, creative_prompt, temperature=0.2, **extra_kwargs)
        print(f"\n{name}:")
        print(r["response"])
    except Exception as e:
        print(f"  ❌ {name}: {e}")

print(f"\n\n🔥 HIGH TEMPERATURE (1.0) — Creative / Diverse")
print("=" * 60)
for name, fn, extra_kwargs in temp_providers:
    try:
        r = fn(creative_system, creative_prompt, temperature=1.0, **extra_kwargs)
        print(f"\n{name}:")
        print(r["response"])
    except Exception as e:
        print(f"  ❌ {name}: {e}")

🧊 LOW TEMPERATURE (0.2) — Safe / Predictable

OpenAI (gpt-4.1-mini):
1. PayZap: Zap Payments, Zap Hassles, Zap Done!  
2. PayZap: Instant UPI, Infinite Possibilities.  
3. PayZap: Your Wallet’s New Best Friend.

Anthropic (Claude Haiku):
# PayZap Taglines

1. **Zap. Pay. Done. Instant money, zero hassle.**

2. **Lightning-fast payments that actually feel like magic.**

3. **Your money moves at the speed of thought.**

Google Gemini:
Here are 3 catchy taglines for PayZap:

1.  **Pay

Groq (Llama 3.3):
Here are three options:
1. "PayZap: Fast Payments Made Easy"
2. "Zap Your Payments Instantly"
3. "PayZap: Pay Fast, Live Free"


🔥 HIGH TEMPERATURE (1.0) — Creative / Diverse

OpenAI (gpt-4.1-mini):
1. PayZap: Lightning-fast payments, zero hassle.  
2. Tap, Pay, Done — PayZap in a zap!  
3. PayZap: Your UPI, simplified and supercharged.

Anthropic (Claude Haiku):
# PayZap Taglines

1. **Zap. Pay. Done. Lightning-fast UPI transfers.**

2. **Your money moves at lightning speed now.**

3. **O

### 📝 Key Observations
- At **low temperature**, outputs converge — run it twice and you'll likely get very similar taglines
- At **high temperature**, you see more unexpected/creative phrasing, but also higher risk of off-topic output
- Temperature ranges differ by provider: OpenAI (0–2), Anthropic (0–1), Gemini (0–2), Groq/Llama (0–2)
- **Production tip:** Use low temp for factual/structured tasks, higher for creative generation

### 9a. Demonstrating Temperature Consistency (Same Prompt, Multiple Runs)

Let's call the same model 3 times at low vs. high temperature to observe output variance.

In [37]:
# --- Run OpenAI gpt-4.1-mini 3 times at low vs. high temp ---
simple_prompt = "Write one catchy tagline (max 8 words) for a health insurance app called 'CareShield'."

for temp_label, temp_val in [("LOW (0.1)", 0.1), ("HIGH (1.0)", 1.0)]:
    print(f"\n🌡️ Temperature {temp_label}:")
    for run in range(1, 4):
        r = call_openai(creative_system, simple_prompt, model="gpt-4.1-mini", temperature=temp_val)
        print(f"  Run {run}: {r['response'].strip()}")


🌡️ Temperature LOW (0.1):
  Run 1: CareShield: Protecting Health, Empowering Lives.
  Run 2: CareShield: Protecting Health, Empowering Lives.
  Run 3: CareShield: Your Health, Our Shield, Always.

🌡️ Temperature HIGH (1.0):
  Run 1: CareShield: Protecting Health, Simplifying Peace of Mind.
  Run 2: CareShield: Your Health, Our Shield, Always.
  Run 3: CareShield: Protecting Health, Empowering Lives.


Notice how low temperature produces near-identical outputs across runs, while high temperature produces different variations each time.

---
## 10. Task 7 — Configurable Parameters Deep-Dive

### 10a. Reasoning Effort (`gpt-5-mini` only)

Reasoning models like `gpt-5-mini` don't use temperature. Instead, they use `reasoning_effort` to control how deeply they think before answering.

**Business Scenario: Loan Eligibility Assessment**

Let's see how reasoning depth affects the quality and speed of a loan assessment.

In [38]:
# --- Reasoning effort comparison: low vs. medium vs. high ---
loan_system = "You are a bank's credit assessment engine. Provide a clear recommendation."

loan_prompt = """Applicant: Priya Sharma, 32F, Software Engineer at TCS (6 years).
Monthly salary: ₹1.2L | Existing EMIs: ₹15K (car loan, 2 years remaining)
CIBIL Score: 762 | Savings: ₹8L | Requested: Home loan of ₹50L for 20 years.
Assess eligibility and give a recommendation with key factors."""

for effort in ["low", "medium", "high"]:
    r = call_openai(loan_system, loan_prompt, reasoning_effort=effort)
    print(f"\n{'━' * 60}")
    print(f"Reasoning Effort: {effort.upper()} | Latency: {r['latency_sec']}s | Output tokens: {r['output_tokens']}")
    print(f"{'━' * 60}")
    print(r["response"][:500])  # Truncate for readability
    if len(r["response"]) > 500:
        print("...")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Reasoning Effort: LOW | Latency: 10.0s | Output tokens: 448
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Reasoning Effort: MEDIUM | Latency: 7.84s | Output tokens: 448
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Reasoning Effort: HIGH | Latency: 10.61s | Output tokens: 448
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



### 📝 Key Observations
- **Low effort:** Faster, fewer tokens — gives a quick directional answer
- **High effort:** Slower, more thorough — considers edge cases, provides detailed reasoning
- In production, use `low` for simple lookups/classifications, `high` for complex analysis

### 10b. Max Tokens Control

Controlling output length is critical for cost management and UX. Let's see how different `max_tokens` values affect the output.

In [39]:
# --- max_tokens comparison across Groq (fast, easy to iterate) ---
system = "You are a concise financial advisor."
prompt = "Explain mutual funds to a first-time investor in India."

for max_tok in [50, 150, 500]:
    r = call_groq(system, prompt, max_tokens=max_tok, temperature=0.5)
    word_count = len(r["response"].split())
    print(f"\nmax_tokens={max_tok} → output_tokens={r['output_tokens']} | ~{word_count} words")
    print(f"  {r['response'][:200]}{'...' if len(r['response']) > 200 else ''}")


max_tokens=50 → output_tokens=50 | ~41 words
  As a first-time investor in India, mutual funds can be a great way to start. Here's a concise overview:

**What are Mutual Funds?**
Mutual funds are investment vehicles that pool money from many inves...

max_tokens=150 → output_tokens=150 | ~97 words
  As a first-time investor in India, mutual funds are a great way to start. Here's a concise overview:

**What are Mutual Funds?**
Mutual funds are investment vehicles that pool money from many investor...

max_tokens=500 → output_tokens=465 | ~324 words
  As a first-time investor in India, mutual funds can be a great way to start investing. Here's a concise overview:

**What are Mutual Funds?**
Mutual funds are investment vehicles that pool money from ...


---
## 11. Quick Reference — API Patterns Cheat Sheet

| Feature | OpenAI (Responses API) | Anthropic (Messages API) | Gemini (GenAI SDK) | Groq (OpenAI-compat) | AWS Bedrock (Converse API) |
|---|---|---|---|---|---|
| **Client init** | `OpenAI()` | `Anthropic()` | `genai.Client()` | `Groq()` | `boto3.client('bedrock-runtime')` |
| **API call** | `client.responses.create()` | `client.messages.create()` | `client.models.generate_content()` | `client.chat.completions.create()` | `client.converse()` |
| **System prompt** | `instructions=` param | `system=` param | `config.system_instruction` | `{'role': 'system'}` in messages | `system=[{'text': ...}]` |
| **User input** | `input=` (string or list) | `messages=[{role, content}]` | `contents=` | `messages=[{role, content}]` | `messages=[{role, content: [{text}]}]` |
| **Get text** | `response.output_text` | `response.content[0].text` | `response.text` | `response.choices[0].message.content` | `response['output']['message']['content'][0]['text']` |
| **Max output** | `max_output_tokens` | `max_tokens` | `config.max_output_tokens` | `max_tokens` | `inferenceConfig.maxTokens` |
| **Temperature** | ✅ (non-reasoning only) | ✅ (0–1) | ✅ (0–2) | ✅ (0–2) | ✅ (model-dependent) |
| **Reasoning effort** | ✅ (reasoning models) | ❌ | ❌ | ❌ | ❌ |
| **Auth** | `OPENAI_API_KEY` | `ANTHROPIC_API_KEY` | `GEMINI_API_KEY` | `GROQ_API_KEY` | AWS IAM (keys / role) |


---
## 12. Conclusion & Key Takeaways

### What We Covered

| Task | Key Takeaway |
|---|---|
| **Five provider calls** | Each provider has a distinct SDK/API pattern — but the Converse-style message shape is now near-universal (Anthropic, Bedrock, Groq) |
| **Unified wrappers** | A common `(system, user, **kwargs) → dict` interface lets application code swap providers without rewrites |
| **Temperature control** | Low = deterministic; High = creative; reasoning models skip temperature entirely |
| **Reasoning effort** | `gpt-5-mini` uses `reasoning_effort` (low/medium/high) instead of temperature |
| **Max tokens** | Controls output length — critical for both cost and UX |
| **Bedrock — boto3 vs ChatBedrock** | `boto3` for tight Lambdas, `ChatBedrock` when the call sits inside a LangChain chain or agent (Module 3 onward) |

### Production Insights
- **Reasoning vs. non-reasoning models.** Use `gpt-5-mini` for tasks that need analysis; `gpt-4.1-mini` for fast, controllable generation.
- **Provider-agnostic wrappers** are essential — they enable A/B testing and graceful fallbacks without rewrites.
- **Groq + Llama** is excellent for internal tools where latency matters more than peak quality.
- **AWS Bedrock** is the default path for regulated clients (banking, insurance, healthcare) — VPC-private, IAM-controlled, region-pinned. We'll build full pipelines on Bedrock in Module 6.

## 13. Stretch Exercise (Optional)
1. **Add a 6th provider.** Try Mistral on Bedrock (`mistral.mistral-large-2407-v1:0`) or Cohere Command — both via the same `bedrock.converse(...)` call. Observe how only `modelId` changes.
2. **Run your own domain prompts.** Pick legal, manufacturing, or education prompts; compare quality and latency across all 5 providers.
3. **`previous_response_id`.** Use OpenAI's Responses-API `previous_response_id` to thread a 3-turn conversation without manually managing history.
4. **Gemini thinking budget.** Set `thinking_config.thinking_budget` on `gemini-2.5-flash` and observe quality vs latency.
5. **Drop-in `ChatBedrock` swap.** Replace your `call_anthropic` wrapper with one that calls `ChatBedrock` instead — show that the side-by-side comparison cell still runs unchanged.

---

**Next Lab:** Lab 1.2 — Robustness Patterns (Retries, Backoff, Fallback Providers) 🛡️
